# Parameter Explorer

Loading the nidus dataset, inspecting individual parameters,
filtering across axes, and rendering basic statistics.

This is the canonical "first contact" notebook for the nidus
package. It exercises every public API documented in the README.

## Setup

In [ ]:
import nidus
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
ds = nidus.load()
ds

## Inspect a single parameter

In [ ]:
p = ds["maternal_cardiovascular.baseline_cardiac_output_l_per_min"]
print(f"Name:           {p.name}")
print(f"Subsystem:      {p.subsystem}")
print(f"Value:          {p.value.central} {p.value.units}")
print(f"Range:          {p.value.low}–{p.value.high}")
print(f"Distribution:   {p.value.distribution} (CI {p.value.ci})")
print(f"Tier:           {p.tier}")
print(f"Citations:      {len(p.citations)}")
print(f"Primary DOI:    {p.primary_citation.doi}")
print()
print("Tier rationale:")
print(p.tier_rationale)

## Filter by subsystem

``Dataset.filter`` returns a new ``Dataset`` containing the
matching parameters. All citations and tier definitions are
carried through unchanged so back-references still resolve.

In [ ]:
cardio = ds.filter(subsystem="maternal_cardiovascular")
print(f"{len(cardio)} maternal-cardiovascular parameters")
for p in list(cardio)[:5]:
    print(f"  [{p.tier}] {p.id}")

## Filter by tier

In [ ]:
tier_a = ds.filter(tier="A")
print(f"{len(tier_a)} Tier-A parameters across all subsystems")
for p in list(tier_a)[:5]:
    print(f"  {p.id}")

## Combine filters

In [ ]:
b_or_c = ds.filter(subsystem="placental_glucose", tier=["B", "C"])
for p in b_or_c:
    print(f"  [{p.tier}] {p.id}  ({p.value.central} {p.value.units})")

## Tier distribution across the whole dataset

In [ ]:
tier_counts = Counter(p.tier for p in ds)
tier_counts

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
tiers = ["A", "B", "C", "D"]
colors = ["#2ca02c", "#1f77b4", "#ff7f0e", "#d62728"]
ax.bar(tiers, [tier_counts.get(t, 0) for t in tiers], color=colors)
ax.set_ylabel("Number of parameters")
ax.set_title(f"Tier distribution across {len(ds)} parameters")
plt.tight_layout()
plt.show()

## Compare values within one subsystem

In [ ]:
subset = list(ds.filter(subsystem="fetal_growth"))
labels = [p.name for p in subset]
centrals = [p.value.central for p in subset]
units = [p.value.units for p in subset]
for label, central, unit in zip(labels, centrals, units):
    print(f"  {central:>8.3g} {unit:<14s}  {label}")

## What next

- **[02_tier_walkthrough.ipynb](02_tier_walkthrough.ipynb)** —
  what the A/B/C/D tiers mean and how they propagate through
  derived quantities.
- **[03_uncertainty_propagation.ipynb](03_uncertainty_propagation.ipynb)** —
  composing parameters across subsystems with explicit
  uncertainty handling.
- **[04_citation_provenance.ipynb](04_citation_provenance.ipynb)** —
  citation graph, load-bearing papers, BibTeX export.